The goal of this TD is to code and play with [PointNet](https://arxiv.org/pdf/1612.00593.pdf), a standard architecture for deep learning for pointcloud.

In this lab, we will ask you to provide both:
- notebook with logs (training logs in particular)
- the weights of the trained networks (classifier and segmentation). We provide guidance for saving the logs directly in your google drive, you can also download them via the colab interface

It should be helpful to use the slides on the side of this TD to obtain additionnal illustrations.

The idea of PointNet is to learn functions which are by definition fit to process unordered set of elements, that is functions of the shape
$$
f\big(\left\{x_1,\dots, x_n\right\}\big) = g\big(h(x_1),\dots, h(x_n)\big)
$$

where $g$ is a *symmetric* function, such as the $\max$ function.

In this TD, we will build the network as presented in the following image (slide 34):
![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD4/PointNet_architecture.PNG)

It consists in:
1. An input / feature transformation module
3. Per-vertex MLP
4. Feature aggregation for classification

In [1]:
try:  # Colab only (widgets for the 3D mesh plots); skipped when running locally
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import os
import numpy as np
import matplotlib.pyplot as plt

import torch.nn as nn
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader

# 1. PointNet Feature Extractor

## 1.1 Input and feature Transformation

The feature transformation module takes as input a $d$ dimensional point-cloud and regresses a $d\times d$ transformation matrix $T$.

If the input consists in $[x,y,z]$ coordinates, we can expect the network to learn a $3\times 3$ matrix which for instance rotates the point cloud in a *canonical* orientation.

If the input consists in $d$-dimensional features, the transformation matrix is harder to visualize.

![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD4/feature_transform.PNG)

This is done in two steps:
1. Apply per-vertex MLP to obtain a $d_{out}$ dimensional embedding for each vertex
2. use MaxPooling over the first dimension to to obtain a simple $d_{out}$ dimensional vector **for the whole point cloud**
3. Use MLP to obtain a $d^2$ dimensional vector, which defines T (flattened).
---


### Question 1

Implement the `FeatTransform` module.

**NOTE :**

Except for the **last** linear layer of the second MLP, all layers consist in of the combination of `layer`, `Batchnorm` and `ReLu` activations. The last layer is a simple linear layer with **no activation** or batchnorm (we don't want to constraint the values of matrix T).

An important (and annoying) feature in [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) is that is expects the features to lie on the **second** dimension (the one after the batch). On the contrary fully connected layers expect features to lie on the **last** dimension. Unless the input is 2d, we would need to reshape values for each layer. Hopefully there is a small trick to handle this.

The trick consist in using [`nn.Conv1d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) with kernel size of $1$ to define fully connected layers. This is exactly equivalent to fully connected layers although a bit surprising at first sight. The adventage is that torch convolutions expect features to be on the **second** dimension.

Follow the following guide (or reshape at every layer, as you wish):

1. For the first MLP, the input is given with shape `(B,d,n)`. The features are in the second dimension, so use [`nn.Conv1d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html), which allows to use [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) directly without reshaping.
2. In the second layer you can give as input a tensor of shape `(B,d_out)` (instead of `(B,1,d_out)`). Both  [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) and [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) will work since the **second** and **last** dimensions are the same.
3. Check the documentation of [`torch.max`](https://pytorch.org/docs/stable/generated/torch.max.html), as it does not return the same thing as numpy (use `torch.max(a,dim=dim).values`).
4. We add an argument to potentially remove batchnorm (for future experiments using batch size of 1)


To stabilize training, we actually output $I + T$ so that at initialization when $T\simeq 0$ this module has no effect.


In [45]:
def make_mlp(channels, use_bn=True, last_activation=True):
    layers = []
    pairs = list(zip(channels[:-1], channels[1:]))
    for i, (c_in, c_out) in enumerate(pairs):
        layers.append(nn.Conv1d(c_in, c_out, 1))
        if last_activation or i < len(pairs) - 1:
            if use_bn:
                layers.append(nn.BatchNorm1d(c_out))
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


In [46]:
class FeatTransform(nn.Module):
    # TODO
    def __init__(self, inp_dim=3, hidden_dims1=[64,128,1024], hidden_dims2=[512,256], use_bn=True):
      """
      inp_dim : int - dimension d of the input
      hidden_dims1 : list - hidden layers (inluding d_out) of the first MLP block (defined with nn.Conv1d...)
      hidden_dims2 : list - hidden layers (without d_out) of the second MLP block
      use_bn       : bool - whether to use batchnorm
      """
      super().__init__()

      ## TODO
      self.inp_dim = inp_dim
      self.use_bn = use_bn
      self.MLP_1 = make_mlp([inp_dim] + hidden_dims1, use_bn=use_bn, last_activation=True)
      '''self.MLP_1 = torch.nn.Sequential(
          torch.nn.Conv1d(self.inp_dim, hidden_dims1[0], 1),
          torch.nn.BatchNorm1d(hidden_dims1[0]),
          torch.nn.ReLU(),
          torch.nn.Conv1d(hidden_dims1[0], hidden_dims1[1], 1),
          torch.nn.BatchNorm1d(hidden_dims1[1]),
          torch.nn.ReLU(),
          torch.nn.Conv1d( hidden_dims1[1],  hidden_dims1[2], 1),
          torch.nn.BatchNorm1d(hidden_dims1[2]),
          torch.nn.ReLU(),
      )'''

      self.MLP_2 = make_mlp([hidden_dims1[-1]] + hidden_dims2 + [inp_dim * inp_dim],
                      use_bn=False, last_activation=False)
      '''self.MLP_2 = torch.nn.Sequential(
          torch.nn.Conv1d(hidden_dims1[2], hidden_dims2[0], 1),
          torch.nn.BatchNorm1d(hidden_dims2[0]),
          torch.nn.ReLU(),
          torch.nn.Conv1d(hidden_dims2[0], hidden_dims2[1], 1),
          torch.nn.BatchNorm1d(hidden_dims2[1]),
          torch.nn.ReLU(),
          torch.nn.Conv1d(hidden_dims2[1], inp_dim*inp_dim, 1),
      )'''




    def forward(self, x):
        """
        x : (B, d, n) - This is standard shape for convolution inputs

        Output
        ------------
        (B, d , d) : output T defined as I + NET(x) for stability
        """
        a1 = self.MLP_1(x)
        a1_pool = torch.max(a1, dim = 2, keepdim=True).values
        T = self.MLP_2(a1_pool).view(x.shape[0], self.inp_dim, self.inp_dim)


        return T + torch.eye(self.inp_dim, device=T.device).unsqueeze(0)

In [47]:
# Try a forward pass
FeatTransform()(torch.rand(2,3,10))

tensor([[[ 1.2623, -0.2530,  0.3877],
         [ 0.0975,  0.8561,  0.1247],
         [ 0.1087, -0.3570,  1.1316]],

        [[ 1.3236, -0.1857,  0.2515],
         [ 0.0558,  0.8295,  0.0612],
         [ 0.0985, -0.4135,  1.1091]]], grad_fn=<AddBackward0>)

## 1.2 PointNet Feature extractor

We now build the PointNet feature extractor network (output is `global_feature`)
![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD4/PointNet_architecture.PNG)


It can be described as follows:
1. Predict first feature transform and apply to the input
2. Apply a first 2-layer MLP with BatchNorm + ReLu.
3. Preduct second feature transform and apply to the features
4. Apply a second MLP with Batchnorm + ReLu, **except** on last layer where there is no activation (`maxpool`will serve as non-linearity).
5. Apply Maxpooling to obtain the global feature vector

Note that for future segmentation task, we will also need to outpu the second transformed features (or the input to the second MLP).

---

### Question 2
Implement the `PointNetfeat` module with the following convention

1. Use again [`nn.Conv1d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) for both MLP blocks.
2. The **first** MLP is the 2-layer MLP `mlp (64,64)` of the picture: both layers output `mlp2_hidden_dims[0]` (=64) features, which is also the input size of the second MLP.
3. The module returns the transformed features before the are fed to the second MLP, the transformation matrix of the second block, and the global feature.
4. Both Feature Transform modules use the same hidden dimensions for their MLPs.
5. You can hardcode to default parameters, we will never change them


In [48]:
class PointNetfeat(nn.Module):

    ## TODO
    def __init__(self, inp_dim=3, mlp2_hidden_dims=[64,128,1024], transf_hidden_dims1=[64,128,1024], transf_hidden_dims2=[512,256], use_bn=True):
      """
      inp_dim   : int - dimension d of the input
      mlp2_hidden_dims : list - hidden layers (inluding d_out) of the second MLP block (defined with nn.Conv1d...)
      transf_hidden_dims1 : list - hidden layers (inluding d_out) of the first mlp in each FeatureTransform block
      transf_hidden_dims2 : list - hidden layers (without d_out) of the second mlp in each FeatureTransform block
      use_bn : bool - whether to use batchnorm
      """
      super().__init__()

      # USEFULE INFO
      self.inp_dim = inp_dim
      self.latent_dim1 = mlp2_hidden_dims[0] # size of the second features
      self.latent_dim = mlp2_hidden_dims[-1] # is also the out_dim
      self.use_bn = use_bn

      self.T1 = FeatTransform(inp_dim, transf_hidden_dims1, transf_hidden_dims2, use_bn)
      self.MLP_1 = make_mlp([inp_dim, self.latent_dim1, self.latent_dim1], use_bn=use_bn, last_activation=True)
      self.T2 = FeatTransform(self.latent_dim1, transf_hidden_dims1, transf_hidden_dims2, use_bn)
      self.MLP_2 = make_mlp([self.latent_dim1] + mlp2_hidden_dims, use_bn=use_bn, last_activation=False)
      '''self.MLP_1 = torch.nn.Sequential(
          torch.nn.Conv1d(inp_dim, self.latent_dim1,1),
          torch.nn.BatchNorm1d(self.latent_dim1),
          torch.nn.ReLU(),
          torch.nn.Conv1d(self.latent_dim1, self.latent_dim1,1),
          torch.nn.BatchNorm1d(self.latent_dim1),
          torch.nn.ReLU(),
      )'''


      '''self.MLP_2 = torch.nn.Sequential(
        torch.nn.Conv1d(self.latent_dim1, mlp2_hidden_dims[0],1),
        torch.nn.BatchNorm1d(mlp2_hidden_dims[0]),
        torch.nn.ReLU(),
        torch.nn.Conv1d(mlp2_hidden_dims[0], mlp2_hidden_dims[1],1),
        torch.nn.BatchNorm1d(mlp2_hidden_dims[1]),
        torch.nn.ReLU(),
        torch.nn.Conv1d(mlp2_hidden_dims[1], mlp2_hidden_dims[2],1),
      )'''



    def forward(self, x):
      """
      Parameters
      --------------
      x : (B, in_dim, n) input batch in convolution format.

      Output
      -------------
      global_feature : (B, d_out) output global feature (d_out=1024)
      T_feat : (B, latent1, latent1) transformation matrix of the second FeatureTransform module
      x_feat : (B, latent1, n) per-point features after being transformed by the second
                FeatureTransform module
      """
      T1 = self.T1(x)
      a1 = self.MLP_1(torch.bmm(T1, x))
      T_feat = self.T2(a1)
      x_feat = torch.bmm(T_feat, a1)
      out = self.MLP_2(x_feat)
      global_feature = torch.max(out, dim = 2, keepdim=True).values

      return global_feature, T_feat, x_feat

In [49]:
# Try a forward pass
global_, T_, x_ = PointNetfeat()(torch.rand(2,3,2048))
print(global_.shape, T_.shape, x_.shape) #check the shapes

torch.Size([2, 1024, 1]) torch.Size([2, 64, 64]) torch.Size([2, 64, 2048])


# 2 PointNet Classifier

## 2.1 Network Design

The PointNet Classifier is the complete network seen as follow
![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD4/PointNet_architecture.PNG)


### Question 3
Implement the `PointNetCls` module.
1. The last MLP also uses BatchNorm + ReLu, **except** for the last layer, which uses either `log_softmax` directly or `softmax`, or even nothing. Note that the training loss you'll chose will be adapted to this choice !
2. You can hardcode to default parameters, we will not change them
3. We don't need the transformed feature, only the transformation matrix for classification

In [50]:
class PointNetCls(nn.Module):
    ## TODO
    def __init__(self, n_cls, inp_dim=3, cls_hidden_dims=[512,256], mlp2_hidden_dims=[64,128,1024], transf_hidden_dims1=[64,128,1024], transf_hidden_dims2=[512,256], use_bn=True):
        """
        PointNet classification network

        Parameters
        ------------------
            n_cls (int): Number of classes for classification
            inp_dim (int): Dimension of the input points
            cls_hidden_dims (list): List of hidden dimensions for the MLP classifier
            mlp2_hidden_dims (list): List of hidden dimensions for the second MLP block in the feature extractor
            transf_hidden_dims1 (list): List of hidden dimensions for the first MLP in each FeatureTransform block
            transf_hidden_dims2 (list): List of hidden dimensions for the second MLP in each FeatureTransform block
            use_bn (bool): Whether to use batchnorm
        """

        super(PointNetCls, self).__init__()

        self.use_bn = use_bn

        ## TODO

        self.feat_extractor = PointNetfeat(inp_dim, mlp2_hidden_dims, transf_hidden_dims1, transf_hidden_dims2,use_bn)


        self.MLP_classifier = make_mlp(channels = [mlp2_hidden_dims[-1], 512, 256, n_cls], last_activation=False)
        '''self.MLP_classifier = torch.nn.Sequential(
            torch.nn.Conv1d(mlp2_hidden_dims[-1], 512,1),
            torch.nn.BatchNorm1d(512),
            torch.nn.ReLU(),
            torch.nn.Conv1d(512, 256,1),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Conv1d(256, n_cls,1),
            torch.nn.LogSoftmax(dim=1)

        )'''


    def forward(self, x):
      """
      Parameters
      --------------
      inp : (B, n, d) input batch in standard format.

      Output
      -------------
      out : (B, n_cls) output score for each class
      T_feat : (B, latent1, latent1) transformation matrix of the second FeatureTransform module
      """
      x = x.transpose(2,1) # Put input in convolution format
      global_feature, T_feat, x_feat = self.feat_extractor(x)
      out = self.MLP_classifier(global_feature)

      if out.dim() == 3:
        out = out.squeeze(-1)

      return out, T_feat

## 2.2 - Data Augmentation

Since PointNet takes $(X,Y,Z)$ coordinates as input, it is not invariant to rigid transformations of the point cloud. One possibility is to make it learn to be invariant, by rigidly modifying shapes during training.

### Question 4
Fill the random scaling and translation function using numpy.
1. Scale and translate each coordinate independantly
2. Scaling factor lies between $\frac{2}{3}$ and $\frac{3}{2}$
2. Translation lies between $-0.2$ and $0.2$

In [51]:
import numpy as np

In [52]:
def random_scale_transl(X):
        """Apply random scaling to the point cloud.
        Scaling is applied for

        Args:
            X (np.ndarray): Point cloud data, (N, 3).

        Returns:
            X_transformed (np.ndarray): Scaled (and translated) point cloud data (N, 3).
        """

        return X_transformed

### Question 5
For this specfific dataset, all shapes pre-aligned, and only variability is rotation along the $y$ axis.

Therefore simply generate a random rotation matrix along the y axis.

In [53]:
def random_rotation(X):
        # TODO
        """Apply random rotation to the point cloud (along the y axis)

        Args:
            X (np.ndarray): Point cloud data, (N, 3).

        Returns:
            X_rot (np.ndarray): Rotated point cloud data (N, 3).
        """

        return X_rot

## 2.3 Data Loading

We will train on the [ModelNet40](https://www.cv-foundation.org/openaccess/content_cvpr_2015/papers/Wu_3D_ShapeNets_A_2015_CVPR_paper.pdf) dataset.

This dataset consist in pointclouds of 12000 objects of 40 different categories such as airplane, car, plant, lamp. Because we only need to provide a global feature vector for each shape, we can subsample each surface with a fixed number of parameters, which allows us to use large batch sizes.

![](https://production-media.paperswithcode.com/datasets/modelnet.jpeg)


We provide below code for dataloading.

Let's first download the data and define the loaders.

In [11]:
!mkdir 'data'
!wget -O './data/modelnet40_ply_hdf5_2048.zip' https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip?download=true --no-check-certificate
!unzip data/modelnet40_ply_hdf5_2048.zip -d ./data/

--2026-10-24 16:39:20--  https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip?download=true
Resolving huggingface.co (huggingface.co)... 3.171.171.6, 3.171.171.128, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.6|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64e358ce030431c0c66908e9/dc5087230fecaf9509fc732b2bc513daa680aa6077d8edb7e05eb5c8d83db427?response-content-type=application%2Fzip&xip=002PaGL3fTI&X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27modelnet40_ply_hdf5_2048.zip%3B+filename%3D%22modelnet40_ply_hdf5_2048.zip%22%3B&Expires=1792863560&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRlMzU4Y2UwMzA0MzFjMGM2NjkwOGU5L2RjNTA4NzIzMGZlY2FmOTUwOWZjNzMyYjJiYzUxM2RhYTY4MGFhNjA3N2Q4ZWRiN2UwNWViNWM4ZDgzZGI0MjdcXD9yZXNwb25zZS1jb250ZW50LXR5cGU9KiZ4aXA9

In [54]:
import h5py
import os
import os.path as osp
import numpy as np  # <-- AÑADIDO
import torch
from pathlib import Path
from glob import glob
from tqdm.auto import tqdm
from torch.utils.data import Dataset


def load_h5(h5_filename):
    f = h5py.File(h5_filename, 'r')
    print(h5_filename)
    data = f['data'][:]
    label = f['label'][:]
    f.close()
    return (data, label)


def load_h5_files(data_path, files_list_path):
    """Load h5 into memory"""
    files_list = [Path(line.rstrip()).name for line in open(osp.join(data_path, files_list_path))]
    data = []
    labels = []
    for i in range(len(files_list)):
        data_, labels_ = load_h5(os.path.join(data_path, files_list[i]))
        data.append(data_)
        labels.append(labels_)
    data = np.concatenate(data, axis=0)
    labels = np.concatenate(labels, axis=0)
    return data, labels


# --- FUNCIONES DE AUMENTACIÓN AÑADIDAS ---

def rotate_point_cloud_y(batch_data):
    """
    Rota aleatoriamente la nube de puntos 3D alrededor del eje Y (eje vertical).
    Input: (N, 3) - Output: (N, 3)
    """
    rotation_angle = np.random.uniform() * 2 * np.pi
    cosval = np.cos(rotation_angle)
    sinval = np.sin(rotation_angle)

    rotation_matrix = np.array([
        [cosval, 0, sinval],
        [0, 1, 0],
        [-sinval, 0, cosval]
    ])

    return np.dot(batch_data, rotation_matrix)


def scale_point_cloud(batch_data, low=0.8, high=1.25):
    """
    Escala aleatoriamente los puntos (zoom in / zoom out).
    Input: (N, 3) - Output: (N, 3)
    """
    scales = np.random.uniform(low, high)
    return batch_data * scales

# -----------------------------------------


class ModelNet40Generator(Dataset):

    def __init__(self, mode, data_dir, files_list, num_classes=40, num_points=1024, rot_aug=False):
        assert mode.lower() in ('train', 'val', 'test')
        assert files_list in ('train_files.txt', 'test_files.txt')
        self.data, labels = load_h5_files(data_dir, files_list)

        self.mode = mode
        self.num_points = num_points
        self.num_classes = num_classes
        self.num_samples = self.data.shape[0]
        self.rot_aug = rot_aug
        self.labels = np.reshape(labels, (-1,))

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx):
        indexes = np.random.permutation(self.data.shape[1])[:self.num_points]
        X = self.data[idx, indexes, ...]
        y = self.labels[idx, ...]
        y_categorical = torch.from_numpy(np.eye(self.num_classes, dtype='uint8')[y]).long()

        if self.mode == 'train' and self.rot_aug:
            # X = self.random_scaling(X)
            X = self.random_rotation_3d(X)

        return torch.from_numpy(X).float(), y_categorical.float()

    def random_scaling(self, X):
        return scale_point_cloud(X)

    def random_rotation_3d(self, X):
        return rotate_point_cloud_y(X)

## 2.4. Classification Loss

PointNet classification loss is defined as the weighted sum of two losses: the classification loss $L_{cls}$, such as cross-entropy, and a regularization loss $L_{reg}$ which regularizes the FeatureTransform output.

In particular, the the regularization loss forces the (second) predicted transformation matrix $T$ to be near-orthogonal (so close to a "rotation"):

$$
L_{reg}(T) = \|T^\top T - I\|_2^2
$$

This loss is averaged along the batch.

This regularization is only applied to the second transformation matrix, that is the one applied to the feature space.

### Question 6
Code the classification loss. Adapt $L_{cls}$ to whatever activation you used in you architecture.

In [55]:
def classification_loss(preds, gt_labels, T_feat, w_reg):
        """
        Classification loss for PointNet

        Parameters
        --------------
        preds : (B, n_cls) - output of the classifier
        gt_labels : (B,) - ground truth labels
        T_feat : (B, latent1, latent1) - transformation matrix of the second FeatureTransform module
        w_reg : weight of the regularization term

        Output
        -------------
        loss : classification loss L_{cls} + w_reg * L_{reg}
        """


        l_cls = F.cross_entropy(preds, gt_labels)

        B, K, _ = T_feat.shape
        I = torch.eye(K, device=T_feat.device)
        diff = torch.bmm(T_feat.transpose(1, 2), T_feat) - I
        l_reg = torch.mean(torch.sum(diff ** 2, dim=(1, 2)))

        return l_cls + w_reg * l_reg

## 2.5 Training

This cell defines the trainer class, using your classification loss, simply run it.

In [56]:
class Trainer(object):

    def __init__(self, train_loader, valid_loader, device='cuda',
                 lr=1e-3, weight_decay=1e-4, num_epochs=200,
                 lr_decay_every = 50, lr_decay_rate = 0.5,
                 log_interval=10, save_dir=None, **model_cfg):

        """
        train_loader: (torch.utils.DataLoader) DataLoader for training set
        valid_loader: (torch.utils.DataLoader) DataLoader for validation set
        device: (str) 'cuda' or 'cpu'
        lr: (float) learning rate
        weight_decay: (float) weight decay for optimiser
        num_epochs: (int) number of epochs
        lr_decay_every: (int) decay learning rate every this many epochs
        lr_decay_rate: (float) decay learning rate by this factor
        log_interval: (int) print training stats every this many iterations
        save_dir: (str) directory to save model checkpoints
        model_cfg: (dict) keyword arguments for model
        """
        self.model = self.get_model(model_cfg).to(device)
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.device = device
        self.lr = lr
        self.weight_decay = weight_decay
        self.num_epochs = num_epochs

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        self.lr_decay_every = lr_decay_every
        self.lr_decay_rate = lr_decay_rate
        self.log_interval = log_interval
        self.save_dir = save_dir
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)

        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []

        self.inp_feat = model_cfg.get('inp_feat', 'xyz')
        self.num_eig = model_cfg.get('num_eig', 128)
        if not self.inp_feat in ['xyz', 'xyzn', 'hks', 'wks']:
            raise ValueError('inp_feat must be one of xyz, xyzn, hks, wks')

        self.model.to(self.device)

    def get_model(self, model_cfg):
        if model_cfg['name'].lower() == 'pointnet':
            return PointNetCls(n_cls=model_cfg['n_cls'],
                              inp_dim=model_cfg['inp_dim'],
                              cls_hidden_dims=model_cfg['cls_hidden_dims'],
                              mlp2_hidden_dims=model_cfg['mlp2_hidden_dims'],
                              transf_hidden_dims1=model_cfg['transf_hidden_dims1'],
                              transf_hidden_dims2=model_cfg['transf_hidden_dims2'])
        else:
            raise ValueError('%s must be one of PointNet, PointNet++'%model_cfg['name'])

    def adjust_lr(self):
        self.lr = self.lr * self.lr_decay_rate
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.lr

    def forward_step(self, inp):
        """
        Perform a forward step of the model.

        Args:
            inp (torch.Tensor): (N, D) tensor of input features

        Returns:
            pred (torch.Tensor): (N, p_out) tensor of predicted labels.
        """
        inp = inp.to(self.device)
        preds, trans_feat = self.model(inp)
        return preds, trans_feat


    def get_loss(self, preds, gt_labels, trans_feat):
        if gt_labels.ndim > 1 and gt_labels.shape[1] > 1:
            target = gt_labels.argmax(dim=1)
        else:
            target = gt_labels.squeeze().long()

        loss = classification_loss(preds, target, trans_feat, 1e-3)
        return loss

    def train_epoch(self):
        train_loss = 0
        train_acc = 0
        for i, (inp_pts, gt_labels) in enumerate(tqdm(self.train_loader)):
            self.optimizer.zero_grad()
            gt_labels = gt_labels.to(self.device)
            preds, trans_feat = self.forward_step(inp_pts)
            loss = self.get_loss(preds, gt_labels, trans_feat)
            loss.backward()
            self.optimizer.step()
            train_loss += loss.item()
            pred_labels = torch.max(preds, dim=1).indices
            this_correct = pred_labels.eq(gt_labels.argmax(dim=-1)).sum().item()
            train_acc += this_correct/gt_labels.shape[0]

        return train_loss/len(self.train_loader), train_acc/len(self.train_loader)

    @torch.no_grad()
    def valid_epoch(self):
        self.model.eval()
        val_loss = 0
        val_acc = 0
        for i, (inp_pts, gt_labels) in enumerate(tqdm(self.valid_loader, leave=False)):
            gt_labels = gt_labels.to(self.device)
            preds, trans_feat = self.forward_step(inp_pts)
            loss = self.get_loss(preds, gt_labels, trans_feat)
            val_loss += loss.item()
            pred_labels = torch.max(preds, dim=1).indices
            this_correct = pred_labels.eq(gt_labels.argmax(dim=-1)).sum().item()
            val_acc += this_correct/gt_labels.shape[0]

        return val_loss/len(self.valid_loader), val_acc/len(self.valid_loader)

    def run(self):
        for epoch in tqdm(range(self.num_epochs)):
            self.model.train()

            if epoch > 0 and epoch % self.lr_decay_every == 0:
                self.adjust_lr()

            train_ep_loss, train_ep_acc = self.train_epoch()
            self.train_losses.append(train_ep_loss)
            self.train_accs.append(train_ep_acc)

            if epoch % self.log_interval == 0:
                val_loss, val_acc = self.valid_epoch()
                self.val_losses.append(val_loss)
                self.val_accs.append(val_acc)
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_latest.pth'))
                print('Epoch: {:03d}, Train Loss: {:.4f}, Train Acc: {:.2f}%, Val Loss: {:.4f}, Val Acc: {:.2f}%'.format(epoch,
                                                                                                                       train_ep_loss,
                                                                                                                       1e2*train_ep_acc,
                                                                                                                       val_loss,
                                                                                                                       1e2*val_acc))
        torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_final.pth'))

    def test(self):
        file_final = os.path.join(self.save_dir, 'model_final.pth')
        if not os.path.exists(file_final):
            print('-------------------------------------------------------')
            print('No final weights, switching to last weights')
            print('-------------------------------------------------------')
            file_final = os.path.join(self.save_dir, 'model_latest.pth')
        weights = torch.load(file_final)
        self.model.load_state_dict(weights)
        _, score = self.valid_epoch()
        print('Final Valid Accuracy : {:.2f}%'.format(1e2*score))

Run the following cells to train the network.

In [15]:
N_CLS = 40
data_dir = 'data/modelnet40_ply_hdf5_2048'
train_data = ModelNet40Generator(mode='train', data_dir=data_dir, files_list='train_files.txt', rot_aug=True)
val_data = ModelNet40Generator(mode='val', data_dir=data_dir, files_list='test_files.txt')
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=0)

data/modelnet40_ply_hdf5_2048/ply_data_train0.h5
data/modelnet40_ply_hdf5_2048/ply_data_train1.h5
data/modelnet40_ply_hdf5_2048/ply_data_train2.h5
data/modelnet40_ply_hdf5_2048/ply_data_train3.h5
data/modelnet40_ply_hdf5_2048/ply_data_train4.h5
data/modelnet40_ply_hdf5_2048/ply_data_test0.h5
data/modelnet40_ply_hdf5_2048/ply_data_test1.h5


You will need to modify this cell if you are not using Google Colab + Drive

In [16]:
import os
# Comment these if you are going local
from google.colab import drive
drive.mount('/content/drive/')
## Change this if your are going local, otherwise keep it (checkpoints will be in checkpoint folder of your drive)
save_dir = '/content/drive/MyDrive/checkpoints'
#os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive/


In [17]:
model_cfg = dict(name="pointnet", n_cls=N_CLS, inp_dim=3, cls_hidden_dims=[512,256], mlp2_hidden_dims=[64,128,1024], transf_hidden_dims1=[64,128,1024], transf_hidden_dims2=[512,256])

#model_cfg = {'name': 'pointnet', 'n_cls': N_CLS, 'conv_dims': [64, 128, 1024], 'fc_dims': [512, 256]}
trainer = Trainer(train_loader, val_loader, lr=0.001, weight_decay=0.0, num_epochs=100, save_dir=os.path.join(save_dir, 'classifier'), **model_cfg)
trainer.run()

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/78 [00:00<?, ?it/s]

Epoch: 000, Train Loss: 1.6727, Train Acc: 53.66%, Val Loss: 1.5308, Val Acc: 57.77%


  0%|          | 0/308 [00:00<?, ?it/s]

...

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

In [18]:
trainer.test()

  0%|          | 0/78 [00:00<?, ?it/s]

Final Valid Accuracy : 85.02%


# 3. PointNet Segmentation

## 3.1 Network Design

We now build the PointNet segmentation network. The per-point features after the feature transform (`x_feat`, $n\times 64$) are concatenated with the global feature repeated for every point ($n\times 1088$), and a shared MLP outputs class scores for every point:
![](https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD4/pointnet_segmentation.PNG)


### Question 7
Build the segmentation network in the case of a batch size of 1.
**Disable all batchnorms**

In [57]:
class PointNetSeg(nn.Module):
    ## TODO
    def __init__(self, n_cls, inp_dim=3, seg_hidden_dims=[512,256,128], mlp2_hidden_dims=[64,128,1024], transf_hidden_dims1=[64,128,1024], transf_hidden_dims2=[512,256]):
        """
        PointNet classification network

        Parameters
        ------------------
            n_cls (int): Number of classes for classification
            inp_dim (int): Dimension of the input points
            seg_hidden_dims (list): List of hidden dimensions for the segmentation MLP
            mlp2_hidden_dims (list): List of hidden dimensions for the second MLP block in the feature extractor
            transf_hidden_dims1 (list): List of hidden dimensions for the first MLP in each FeatureTransform block
            transf_hidden_dims2 (list): List of hidden dimensions for the second MLP in each FeatureTransform block
        """


        super().__init__()
        self.feat = PointNetfeat(inp_dim, mlp2_hidden_dims, transf_hidden_dims1, transf_hidden_dims2, use_bn=False)
        self.MLP_1 = make_mlp(channels = [mlp2_hidden_dims[0] + mlp2_hidden_dims[-1]] + seg_hidden_dims, use_bn=False,last_activation = True)
        '''self.MLP_1 = torch.nn.Sequential(
            torch.nn.Conv1d(mlp2_hidden_dims[0] + mlp2_hidden_dims[-1], seg_hidden_dims[0], 1),
            torch.nn.BatchNorm1d(seg_hidden_dims[0]),
            torch.nn.ReLU(),
            torch.nn.Conv1d(seg_hidden_dims[0], seg_hidden_dims[1], 1),
            torch.nn.BatchNorm1d(seg_hidden_dims[1]),
            torch.nn.ReLU(),
            torch.nn.Conv1d(seg_hidden_dims[1], seg_hidden_dims[2], 1),
            torch.nn.BatchNorm1d(seg_hidden_dims[2]),
            torch.nn.ReLU(),
        )'''

        self.MLP_2 = make_mlp(channels = [seg_hidden_dims[-1], seg_hidden_dims[-1], n_cls], use_bn=False, last_activation=False)
        '''self.MLP_2 = torch.nn.Sequential(
            torch.nn.Conv1d(seg_hidden_dims[-1], seg_hidden_dims[-1], 1),
            torch.nn.BatchNorm1d(seg_hidden_dims[-1]),
            torch.nn.ReLU(),
            torch.nn.Conv1d(seg_hidden_dims[-1], n_cls, 1),
        )'''


    def forward(self, x):
      """
      Parameters
      --------------
      inp : (B, n, d) input batch in standard format.

      Output
      -------------
      out : (B, n, n_cls) output score for each class
      T_feat : (B, latent1, latent1) transformation matrix of the second FeatureTransform module
      """
      x = x.transpose(2,1) # Put input in convolution format
      global_feature, T_feat, x_feat = self.feat(x)
      global_feature_repeated = global_feature.expand(-1, -1, x.shape[2])
      new_input = torch.cat((x_feat, global_feature_repeated), 1)
      a1 = self.MLP_1(new_input)
      out = self.MLP_2(a1).transpose(2,1)


      return out, T_feat

In [58]:
out_seg, T_feat = PointNetSeg(260)(torch.rand(2, 2046, 3))
print(out_seg.shape, T_feat.shape) # check the shapes

torch.Size([2, 2046, 260]) torch.Size([2, 64, 64])


## 3.2 Data Loading

We will train on the RNA dataset from Lab3. Since all shapes have different number of vertices, we can't use batch size bigger than one.

Run the following cells to download and define code to load the data.

In [30]:
!pip install potpourri3d
!pip install git+https://github.com/skoch9/meshplot.git
!pip install pythreejs

#!wget https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/diffusion_utils.py
#!wget https://www.lix.polytechnique.fr/~pierson/cours/RNADataset.zip
!unzip -o RNADataset.zip
!mkdir models
#!wget https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/material_TD3.zip
!unzip -o material_TD3.zip

  Cloning https://github.com/skoch9/meshplot.git to /tmp/pip-req-build-nbxyzdm2
  Running command git clone --filter=blob:none --quiet https://github.com/skoch9/meshplot.git /tmp/pip-req-build-nbxyzdm2
  Resolved https://github.com/skoch9/meshplot.git to commit 725e4a7926a5f10888f0edd1762fecf9db751c56
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Archive:  RNADataset.zip
   creating: RNADataset/
  inflating: RNADataset/train_full.txt  
  inflating: RNADataset/.DS_Store    
  inflating: __MACOSX/RNADataset/._.DS_Store  
  inflating: RNADataset/LICENSE      
  inflating: RNADataset/test_full.txt  
  inflating: RNADataset/train.txt    
  inflating: RNADataset/label_map    
   creating: RNADataset/labels/
  inflating: RNADataset/rna_image.png  
  inflating: RNADataset/test.txt     
   creating: RNADataset/off/
  inflating: RNADataset/labels/3J8G_A.txt  
  inflating: RNADataset/labels/4V6Z_BB.txt  
  in

In [22]:
!wget -v https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/diffusion_utils.py
!ls

--2026-10-24 17:12:02--  https://www.lix.polytechnique.fr/Labo/Robin.Magnet/INF631/TD3/diffusion_utils.py
Resolving www.lix.polytechnique.fr (www.lix.polytechnique.fr)... 193.55.176.29
Connecting to www.lix.polytechnique.fr (www.lix.polytechnique.fr)|193.55.176.29|:443... failed: No route to host.
data  drive  models  sample_data


In [59]:
from diffusion_utils import normalize_positions, read_mesh
from mesh_utils.mesh import TriMesh
import plot_utils as plu # Comment if you are not visualizing the results

class RNAMeshDataset(Dataset):
    """RNA Mesh Dataset
    """

    def __init__(self, root_dir, train, n_classes=260):

        """
            root_dir (string): Directory with all the meshes.
            train (bool): If True, use the training set, else use the test set.
            num_eig (int): Number of eigenvalues to use.
            op_cache_dir (string): Directory to cache the operators.
            n_classes (int): Number of classes.
        """

        self.train = train  # bool
        self.root_dir = root_dir
        self.n_class = n_classes # (includes -1)

        # store in memory
        self.verts_list = []
        self.faces_list = []
        self.labels_list = []  # per-vertex

        # Load the meshes & labels
        if self.train:
            with open(os.path.join(self.root_dir, "train.txt")) as f:
                this_files = [line.rstrip() for line in f]
        else:
            with open(os.path.join(self.root_dir, "test.txt")) as f:
                this_files = [line.rstrip() for line in f]

        print("loading {} files: {}".format(len(this_files), this_files))

        # Load the actual files

        off_path = os.path.join(root_dir, "off")
        label_path = os.path.join(root_dir, "labels")
        for f in this_files:
            off_file = os.path.join(off_path, f)
            label_file = os.path.join(label_path, f[:-4] + ".txt")

            verts, faces = read_mesh(off_file)
            labels = np.loadtxt(label_file).astype(int) + 1 # shift -1 --> 0

            verts = torch.tensor(verts).float()
            faces = torch.tensor(faces)
            labels = torch.tensor(labels)

            # center and unit scale
            verts = normalize_positions(verts)

            self.verts_list.append(verts)
            self.faces_list.append(faces)
            self.labels_list.append(labels)


    def __len__(self):
        return len(self.verts_list)

    def __getitem__(self, idx):
        return self.verts_list[idx], self.faces_list[idx], self.labels_list[idx]

## 3.3 Segmentation Loss

PointNet segmentation loss is defined the same as the classification loss, except we know perform per-vertex classification using $L_{seg}$ instead of $L_{cls}$. Adapt again the loss to your network design


### Question 8
Code the segmentation loss. Adapt $L_{seg}$ to whatever activation you used in you architecture. You might be able to use exactly the same code.

In [60]:
def segmentation_loss(preds, gt_labels, T_feat, w_reg):
        """
        Classification loss for PointNet

        Parameters
        --------------
        preds : (B, n_cls) - output of the segmentation network
        gt_labels : (B, n) - ground truth labels
        T_feat : (B, latent1, latent1) - transformation matrix of the second FeatureTransform module
        w_reg : weight of the regularization term

        Output
        -------------
        loss : segmentation loss L_{seg} + w_reg * L_{reg}
        """
        l_cls = F.cross_entropy(preds, gt_labels)

        # 2. Pérdida de regularización de PointNet
        B, K, _ = T_feat.shape
        I = torch.eye(K, device=T_feat.device)
        diff = torch.bmm(T_feat.transpose(1, 2), T_feat) - I
        l_reg = torch.mean(torch.sum(diff ** 2, dim=(1, 2)))

        return l_cls + w_reg * l_reg

## 3.4 Training

This defines a trainer class using your loss. Simply run the cells.

In [61]:
class TrainerSeg(object):

    def __init__(self, train_loader, valid_loader, device='cuda',
                 lr=1e-3, weight_decay=1e-4, num_epochs=200,
                 lr_decay_every = 50, lr_decay_rate = 0.5,
                 log_interval=10, save_dir=None, **model_cfg):

        """
        pointNet_cls: (nn.Module) class of the PointNet (or ++) model.
        train_loader: (torch.utils.DataLoader) DataLoader for training set
        valid_loader: (torch.utils.DataLoader) DataLoader for validation set
        device: (str) 'cuda' or 'cpu'
        lr: (float) learning rate
        weight_decay: (float) weight decay for optimiser
        num_epochs: (int) number of epochs
        lr_decay_every: (int) decay learning rate every this many epochs
        lr_decay_rate: (float) decay learning rate by this factor
        log_interval: (int) print training stats every this many iterations
        save_dir: (str) directory to save model checkpoints
        model_cfg: (dict) keyword arguments for model
        """
        self.model = self.get_model(model_cfg).to(device)
        self.train_loader = train_loader
        self.valid_loader = valid_loader
        self.device = device
        self.lr = lr
        self.weight_decay = weight_decay
        self.num_epochs = num_epochs

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        self.loss = torch.nn.CrossEntropyLoss()

        self.lr_decay_every = lr_decay_every
        self.lr_decay_rate = lr_decay_rate
        self.log_interval = log_interval
        self.save_dir = save_dir
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)

        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []

        self.inp_feat = model_cfg.get('inp_feat', 'xyz')
        self.num_eig = model_cfg.get('num_eig', 128)
        if not self.inp_feat in ['xyz', 'xyzn', 'hks', 'wks']:
            raise ValueError('inp_feat must be one of xyz, xyzn, hks, wks')

        self.model.to(self.device)

    def get_model(self, model_cfg):
        if model_cfg['name'].lower() == 'pointnet':
            return PointNetSeg(n_cls=model_cfg['n_cls'],
                              inp_dim=model_cfg['inp_dim'],
                              seg_hidden_dims=model_cfg['seg_hidden_dims'],
                              mlp2_hidden_dims=model_cfg['mlp2_hidden_dims'],
                              transf_hidden_dims1=model_cfg['transf_hidden_dims1'],
                              transf_hidden_dims2=model_cfg['transf_hidden_dims2'])
        else:
            raise ValueError('%s must be one of PointNet, PointNet++'%model_cfg['name'])

    def adjust_lr(self):
        self.lr = self.lr * self.lr_decay_rate
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.lr

    def forward_step(self, inp):
        """
        Perform a forward step of the model.

        Args:
            inp (torch.Tensor): (N, D) tensor of input features

        Returns:
            pred (torch.Tensor): (N, p_out) tensor of predicted labels.
        """
        inp = inp.to(self.device)
        verts = inp[..., :3].to(self.device)
        norms = inp[..., 3:].to(self.device)
        preds, trans_feat = self.model(inp)
        return preds, trans_feat



    def get_loss(self, preds, gt_labels, trans_feat):

        loss = segmentation_loss(preds, gt_labels, trans_feat, 1e-3)
        return loss

    def train_epoch(self):
        train_loss = 0
        train_acc = 0
        for i, (inp_pts, _, gt_labels) in enumerate(tqdm(self.train_loader)):
            self.optimizer.zero_grad()
            gt_labels = gt_labels.to(self.device)
            preds, trans_feat = self.forward_step(inp_pts)
            loss = self.get_loss(preds[0], gt_labels[0], trans_feat)
            loss.backward()
            self.optimizer.step()
            train_loss += loss.item()

            pred_labels = torch.max(preds[0], dim=1).indices  # (N,)
            this_correct = pred_labels.eq(gt_labels).sum().item()
            train_acc += this_correct / inp_pts.shape[-2]

        return train_loss/len(self.train_loader), train_acc/len(self.train_loader)

    @torch.no_grad()
    def valid_epoch(self):
        self.model.eval()
        val_loss = 0
        val_acc = 0
        total = 0
        for i, (inp_pts, _, gt_labels) in enumerate(self.valid_loader):
            gt_labels = gt_labels.to(self.device)
            preds, trans_feat = self.forward_step(inp_pts)
            loss = self.get_loss(preds[0], gt_labels[0], trans_feat)
            val_loss += loss.item()

            pred_labels = torch.max(preds[0], dim=1).indices
            this_correct = pred_labels.eq(gt_labels).sum().item()
            total += inp_pts.shape[-2]
            val_acc += this_correct

        return val_loss/len(self.valid_loader), val_acc/total

    def run(self):
        for epoch in tqdm(range(self.num_epochs)):
            self.model.train()

            if epoch > 0 and epoch % self.lr_decay_every == 0:
                self.adjust_lr()

            train_ep_loss, train_ep_acc = self.train_epoch()
            self.train_losses.append(train_ep_loss)
            self.train_accs.append(train_ep_acc)

            if epoch % self.log_interval == 0:
                val_loss, val_acc = self.valid_epoch()
                self.val_losses.append(val_loss)
                self.val_accs.append(val_acc)
                torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_latest.pth'))
                print('Epoch: {:03d}, Train Loss: {:.4f}, Train Acc: {:.2f}%, Val Loss: {:.4f}, Val Acc: {:.2f}%'.format(epoch,
                                                                                                                       train_ep_loss,
                                                                                                                       1e2*train_ep_acc,
                                                                                                                       val_loss,
                                                                                                                       1e2*val_acc))
        torch.save(self.model.state_dict(), os.path.join(self.save_dir, 'model_final.pth'))

    def visualize(self):
        """
        We only test the first two shapes of validation set.
        """
        self.model.eval()
        test_seg_meshes = []

        for i, (inp_pts, inp_faces, gt_labels) in enumerate(self.valid_loader):
            gt_labels = gt_labels.to(self.device)
            preds, trans_feat = self.forward_step(inp_pts)
            pred_labels = torch.max(preds[0], dim=1).indices
            test_seg_meshes.append([TriMesh(inp_pts.squeeze().cpu().numpy(), inp_faces.squeeze().cpu().numpy()),
                                  pred_labels.squeeze().cpu().numpy()])
            if i==1:
                break

        cmap1 = plt.get_cmap("jet")(test_seg_meshes[0][-1] / (146))[:,:3]
        cmap2 = plt.get_cmap("jet")(test_seg_meshes[1][-1] / (146))[:,:3]

        plu.double_plot(test_seg_meshes[0][0], test_seg_meshes[1][0], cmap1, cmap2)

    def test(self):
        file_final = os.path.join(self.save_dir, 'model_final.pth')
        if not os.path.exists(file_final):
            print('-------------------------------------------------------')
            print('No final weights, switching to last weights')
            print('-------------------------------------------------------')
            file_final = os.path.join(self.save_dir, 'model_latest.pth')
        weights = torch.load(file_final)
        self.model.load_state_dict(weights)
        _, score = self.valid_epoch()
        print('Final Valid Segmentation Accuracy : {:.2f}%'.format(1e2*score))

Run these cells to see

In [62]:
root_dir = './RNADataset/'

train_dataset = RNAMeshDataset(root_dir, train=True)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0, persistent_workers=False)

valid_dataset = RNAMeshDataset(root_dir, train=False)
valid_loader = DataLoader(valid_dataset, batch_size=1, num_workers=0, persistent_workers=False)

loading 60 files: ['5JUO_D.off', '5T62_B.off', '4WCE_Y.off', '4V9K_BB.off', '5E81_16.off', '3J92_7.off', '4V7X_BB.off', '5T7V_C.off', '4TUE_RB.off', '5DGE_3.off', '4V7B_BB.off', '4U4O_3.off', '3JA1_LB.off', '3PIO_Y.off', '5EL7_16.off', '5MDY_3.off', '5H5U_B.off', '4U4O_7.off', '5TBW_AS.off', '5J30_RB.off', '4W4G_RB.off', '4V7J_AB.off', '4V8J_BB.off', '2OTJ_9.off', '4V7X_DB.off', '3J9Y_B.off', '1VQK_9.off', '1Q82_B.off', '4W2H_DB.off', '3G4S_9.off', '1NKW_9.off', '4V5J_BB.off', '3JBN_AB.off', '4V8E_CB.off', '4YZV_YB.off', '4V9P_CB.off', '5J5B_DB.off', '4U6F_7.off', '5OBM_7.off', '3JCD_B.off', '4V55_BA.off', '5IBB_16.off', '1VY7_BB.off', '5GAD_B.off', '4U4Q_3.off', '4WRA_1J.off', '5ADY_A.off', '4V70_BB.off', '4TUB_RB.off', '5AJ0_A4.off', '1KD1_B.off', '4V6N_AA.off', '4V8I_BB.off', '4W4G_YB.off', '4V8F_DB.off', '4V8P_E3.off', '4V5Y_BA.off', '4TUA_YB.off', '5DFE_RB.off', '4V5K_DB.off']
loading 13 files: ['4V83_BB.off', '1VQ6_9.off', '4V7K_BB.off', '4V9A_DB.off', '3JQ4_B.off', '5KCS_1B.off'

In [63]:
#model_cfg = {'name': 'pointnet', 'n_cls': 260, 'conv_dims': [64, 128, 1024], 'fc_dims': [512, 256]}
model_cfg = dict(name="pointnet", n_cls=260, inp_dim=3, seg_hidden_dims=[512,256], mlp2_hidden_dims=[64,128,1024], transf_hidden_dims1=[64,128,1024], transf_hidden_dims2=[512,256])

trainer = TrainerSeg(train_loader, valid_loader, lr=0.001, device='cuda', weight_decay=0.0, num_epochs=300, save_dir=os.path.join(save_dir, 'segmentation'), **model_cfg)
trainer.run()

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Epoch: 000, Train Loss: 4.9529, Train Acc: 1.08%, Val Loss: 4.8204, Val Acc: 1.38%


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

...

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Vizualize the results

In [66]:
trainer.visualize()

TraitError: The 'target' trait of a DirectionalLight instance expected an Uninitialized or an Object3D, not the str 'IPY_MODEL_693e8cae-80cc-475f-ab50-b690e60ff7f7'.

TraitError: The 'target' trait of a DirectionalLight instance expected an Uninitialized or an Object3D, not the str 'IPY_MODEL_265fef4b-3fec-499d-bfa8-3dc4914aeafa'.

TraitError: The 'target' trait of a DirectionalLight instance expected an Uninitialized or an Object3D, not the str 'IPY_MODEL_6d586633-4f8f-463c-9d8f-0f03a731cbfc'.

In [65]:
trainer.test()

Final Valid Segmentation Accuracy : 74.98%
